In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

LABEL_COLUMNS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture"
]


class PerClassLabelRefiner:
    """
    Для каждого класса обучает отдельную модель, используя ТОЛЬКО 
    предсказания этого класса от всех источников.
    
    Например, для ACL:
    Входы: [text_ACL, ens_ACL, text_ACL*ens_ACL, ...]
    Выход: refined_ACL
    
    Никакой информации от других классов не используется!
    """
    
    def __init__(self, model_type='ridge', alpha=1.0, use_interactions=True):
        """
        Parameters:
        -----------
        model_type : str
            'ridge' - Ridge regression
            'logistic' - Logistic regression
        alpha : float
            Параметр регуляризации
        use_interactions : bool
            Добавлять ли взаимодействия между источниками (произведения)
        """
        self.model_type = model_type
        self.alpha = alpha
        self.use_interactions = use_interactions
        self.models = {}  # Словарь моделей: {class_name: model}
        self.scalers = {}  # Словарь скалеров: {class_name: scaler}
        self.is_fitted = False
        self.n_sources = None
        
    def _prepare_features_for_class(self, *label_dfs, class_idx):
        """
        Подготавливает признаки для конкретного класса.
        
        Returns:
        --------
        X : ndarray of shape [n_samples, n_features]
            Признаки только для этого класса от всех источников
        """
        features = []
        
        for df in label_dfs:
            # Берём только колонку текущего класса
            col_values = df[LABEL_COLUMNS[class_idx]].values.astype(float)
            col_values = np.nan_to_num(col_values, nan=0.5)
            features.append(col_values.reshape(-1, 1))
        
        # Если хотим взаимодействия (произведения между источниками)
        if self.use_interactions and len(label_dfs) > 1:
            for i in range(len(label_dfs)):
                for j in range(i+1, len(label_dfs)):
                    col_i = label_dfs[i][LABEL_COLUMNS[class_idx]].values.astype(float)
                    col_j = label_dfs[j][LABEL_COLUMNS[class_idx]].values.astype(float)
                    col_i = np.nan_to_num(col_i, nan=0.5)
                    col_j = np.nan_to_num(col_j, nan=0.5)
                    interaction = (col_i * col_j).reshape(-1, 1)
                    features.append(interaction)
        
        X = np.hstack(features)  # [n_samples, n_features]
        return X
    
    def fit(self, gt_df, *label_dfs):
        """
        Обучает отдельную модель для каждого класса.
        """
        print("🔄 Подготовка данных...")
        
        # Находим строки с GT
        mask_gt_exists = gt_df[LABEL_COLUMNS].notna().any(axis=1)
        gt_labeled_df = gt_df[mask_gt_exists].copy()
        
        print(f"   Найдено {len(gt_labeled_df)} строк с GT лейблами.")
        
        if len(gt_labeled_df) == 0:
            raise ValueError("Нет строк с GT лейблами!")
        
        # Выравниваем все датафреймы по UID
        labeled_uids = gt_labeled_df['StudyInstanceUID'].values
        
        aligned_dfs = []
        for df in label_dfs:
            aligned = df[df['StudyInstanceUID'].isin(labeled_uids)].set_index('StudyInstanceUID').loc[labeled_uids].reset_index()
            aligned_dfs.append(aligned)
        
        gt_aligned = gt_labeled_df.set_index('StudyInstanceUID').loc[labeled_uids].reset_index()
        
        self.n_sources = len(aligned_dfs)
        print(f"   Количество источников: {self.n_sources}")
        
        # Обучаем модель для каждого класса отдельно
        print(f"\n🎯 Обучение моделей для каждого класса...")
        
        for idx, col in enumerate(LABEL_COLUMNS):
            # Подготавливаем признаки ТОЛЬКО для этого класса
            X = self._prepare_features_for_class(*aligned_dfs, class_idx=idx)
            
            # Целевая переменная
            y = gt_aligned[col].values.astype(float)
            
            # Маска валидных лейблов
            valid_mask = ~np.isnan(y)
            
            if valid_mask.sum() < 5 or len(np.unique(y[valid_mask])) < 2:
                print(f"   ⚠️ Пропуск {col}: слишком мало данных или нет вариации")
                continue
            
            X_train = X[valid_mask]
            y_train = y[valid_mask]
            
            # Для Ridge заменяем NaN на среднее (хотя их уже заменили на 0.5)
            if self.model_type == 'ridge':
                # Масштабируем
                scaler = StandardScaler()
                X_train_scaled = scaler.fit_transform(X_train)
                
                # Обучаем Ridge
                model = Ridge(alpha=self.alpha, random_state=42)
                model.fit(X_train_scaled, y_train)
                
                self.models[col] = model
                self.scalers[col] = scaler
                
                print(f"   ✅ {col:20s}: Ridge обучена на {len(y_train)} примерах, признаков: {X_train.shape[1]}")
            
            elif self.model_type == 'logistic':
                # Бинаризуем
                y_train_binary = (y_train > 0.5).astype(int)
                
                scaler = StandardScaler()
                X_train_scaled = scaler.fit_transform(X_train)
                
                model = LogisticRegression(
                    penalty='l2',
                    C=1.0/self.alpha,
                    solver='lbfgs',
                    max_iter=1000,
                    class_weight='balanced',
                    random_state=42
                )
                
                try:
                    model.fit(X_train_scaled, y_train_binary)
                    self.models[col] = model
                    self.scalers[col] = scaler
                    print(f"   ✅ {col:20s}: Logistic обучена на {len(y_train)} примерах")
                except Exception as e:
                    print(f"   ❌ {col:20s}: ошибка - {e}")
        
        self.is_fitted = True
        
        # Валидация
        self._validate_on_gold(gt_aligned, *aligned_dfs)
        
        return self
    
    def _validate_on_gold(self, gt_df, *label_dfs):
        """Валидирует результат на золотых примерах"""
        print("\n📈 Валидация на золотых примерах:")
        
        gt_labels = gt_df[LABEL_COLUMNS].values.astype(float)
        auc_per_class = {}
        baseline_auc = {}
        
        for idx, col in enumerate(LABEL_COLUMNS):
            if col not in self.models:
                continue
            
            y_true = gt_labels[:, idx]
            valid = ~np.isnan(y_true)
            
            if valid.sum() < 5 or len(np.unique(y_true[valid])) < 2:
                continue
            
            # Получаем предсказания модели
            X = self._prepare_features_for_class(*label_dfs, class_idx=idx)
            X_scaled = self.scalers[col].transform(X)
            
            if self.model_type == 'ridge':
                y_pred = self.models[col].predict(X_scaled)
                y_pred = np.clip(y_pred, 0, 1)
            else:  # logistic
                y_pred = self.models[col].predict_proba(X_scaled)[:, 1]
            
            # Baseline: простое усреднение источников
            sources_vals = []
            for df in label_dfs:
                val = df[LABEL_COLUMNS[idx]].values.astype(float)
                val = np.nan_to_num(val, nan=0.5)
                sources_vals.append(val)
            y_baseline = np.mean(sources_vals, axis=0)
            
            try:
                auc_refined = roc_auc_score(y_true[valid], y_pred[valid])
                auc_base = roc_auc_score(y_true[valid], y_baseline[valid])
                
                auc_per_class[col] = auc_refined
                baseline_auc[col] = auc_base
                
                improved = auc_refined - auc_base
                sign = "↑" if improved > 0 else "↓"
                print(f"   {col:20s}: baseline={auc_base:.4f}, refined={auc_refined:.4f} {sign}{abs(improved):.4f}")
            except Exception as e:
                print(f"   {col:20s}: ошибка - {e}")
        
        if auc_per_class:
            mean_auc = np.mean(list(auc_per_class.values()))
            mean_baseline = np.mean(list(baseline_auc.values()))
            improvement = mean_auc - mean_baseline
            sign = "↑" if improvement > 0 else "↓"
            
            print(f"\n   🎯 Средний AUC baseline:  {mean_baseline:.4f}")
            print(f"   🎯 Средний AUC refined:   {mean_auc:.4f}")
            print(f"   📊 Улучшение:             {sign}{abs(improvement):.4f}")
        
        return auc_per_class
    
    def transform(self, *label_dfs):
        """
        Применяет трансформацию к новым данным.
        """
        if not self.is_fitted:
            raise ValueError("Модель не обучена! Сначала вызовите fit().")
        
        print("🔄 Переоценка предсказаний...")
        
        # Выравниваем по UID
        uids = label_dfs[0]['StudyInstanceUID']
        for df in label_dfs[1:]:
            uids = pd.merge(pd.DataFrame({'StudyInstanceUID': uids}), 
                           df[['StudyInstanceUID']], 
                           on='StudyInstanceUID')['StudyInstanceUID']
        
        aligned_dfs = []
        for df in label_dfs:
            aligned = df[df['StudyInstanceUID'].isin(uids)].set_index('StudyInstanceUID').loc[uids].reset_index()
            aligned_dfs.append(aligned)
        
        # Для каждого класса получаем предсказания
        refined_probs = np.zeros((len(uids), len(LABEL_COLUMNS)))
        
        for idx, col in enumerate(LABEL_COLUMNS):
            if col in self.models:
                X = self._prepare_features_for_class(*aligned_dfs, class_idx=idx)
                X_scaled = self.scalers[col].transform(X)
                
                if self.model_type == 'ridge':
                    y_pred = self.models[col].predict(X_scaled)
                    y_pred = np.clip(y_pred, 0, 1)
                else:  # logistic
                    y_pred = self.models[col].predict_proba(X_scaled)[:, 1]
                
                refined_probs[:, idx] = y_pred
            else:
                # Если модель не обучена, берём среднее из источников
                sources_vals = []
                for df in aligned_dfs:
                    val = df[LABEL_COLUMNS[idx]].values.astype(float)
                    val = np.nan_to_num(val, nan=0.5)
                    sources_vals.append(val)
                refined_probs[:, idx] = np.mean(sources_vals, axis=0)
        
        # Создаем результирующий DataFrame
        result_df = aligned_dfs[0][['StudyInstanceUID']].copy()
        result_df[LABEL_COLUMNS] = refined_probs
        
        print("✅ Переоценка завершена.")
        return result_df


# ============================================================
# ИСПОЛЬЗОВАНИЕ
# ============================================================

print("="*60)
print("Загрузка данных...")
print("="*60)

gt_df = pd.read_csv("data/high_agreement_validation.csv")       
df = pd.read_csv("data/llm_labels_v4_blend.csv") 


df1 = pd.read_csv("data/FINAL2_labels_scores.csv") 
df2 = pd.read_csv("data/HYBRID_labels_scores.csv") 
df3 = pd.read_csv("data/consensus_labels.csv") 
df4 = pd.read_csv("data/labels_v2_repaired.csv")
df9 = pd.read_csv("data/report_labels_gpt56sol.csv")


df5 = pd.read_csv("data/pseudo/predictions.csv") 
df6 = pd.read_csv("data/pseudo/raptor_raw.csv") 
df7 = pd.read_csv("data/pseudo/raptor_sub.csv") 
df8 = pd.read_csv("data/pseudo/submission2.csv") 


sources = [df, df1, df2, df3, df4, df5, df6, df7, df8] 


refiner = PerClassLabelRefiner(
    model_type='ridge', 
    alpha=1.0,
    use_interactions=True  # Добавить произведения text_ACL * ens_ACL
)

refiner.fit(gt_df, *sources)

print("\n" + "="*60)
print("Применение ко всем данным")
print("="*60)

refined_df = refiner.transform(*sources)
refined_df.to_csv("data/refined_predictions_v11.csv", index=False)
